# NHS Workforce Data Aggregation and Preparation
This page demonstrates loading, cleaning, and aggregation for NHS Leavers, Hires, and Sickness time series. All steps are clearly documented for clarity and reproducibility.

## 1. 📦 Data Import and Setup

**Note:**  
- Data files are loaded from the NHS shared network directory.
- Output and intermediate results are written to the user's “Desys Work” folder - Alter this to fit your path/source.


In [1]:
import os

import numpy as np
import pandas as pd

## 2. 📥 Load Workforce Data
- Date columns ("Termination Date", "Latest Start Date", "Month") are parsed with day-first or custom formats to prevent ambiguity.
- All columns are lowercased and whitespace-trimmed for safety in grouping and analysis.

## 3. ⚙️ Standardize Column Names and Output Helper
Tip: 
Always standardise DataFrame column names right after import to prevent downstream bugs especially important when joining or grouping by column names with inconsistent casing or stray whitespace!


In [2]:
path = r"S:\WP Personnel\A - Workforce\Data science student\Desys Corner"

leavers = pd.read_csv(
    f"{path}\\NHS_Leavers_Detail_New.csv",
    parse_dates=["Termination Date"],
    dayfirst=True,
)
leavers["Month"] = leavers["Termination Date"].dt.to_period("M").dt.to_timestamp()

hires = pd.read_csv(
    f"{path}\\Hires_Detail_New.csv", parse_dates=["Latest Start Date"], dayfirst=True
)
hires["Month"] = hires["Latest Start Date"].dt.to_period("M").dt.to_timestamp()

sickness = pd.read_csv(f"{path}\\Sickness_Percentage_New.csv")
sickness["Month"] = pd.to_datetime(
    sickness["Month"], format="%b-%y", errors="coerce"
)  # Fix date parsing 'Apr-23' format

# Column names to lowercase and strip whitespace
# Strip whitespace from all column names
leavers.columns = leavers.columns.str.lower().str.strip()
hires.columns = hires.columns.str.lower().str.strip()
sickness.columns = sickness.columns.str.lower().str.strip()

In [3]:
# Set output directory + helper for safe filenames
base_dir = r"C:\Users\Destinee.HassanBien\Documents\Desys Work"
os.makedirs(base_dir, exist_ok=True)


def safe_filename(label):
    return re.sub(r'[\\/:"*?<>| ]+', "_", label)

## 4. 🏷️ Grouping and Monthly Aggregation
Why group at this level?

Grouping by ["service group", "department", "organisation", "staff group"] + "month" enables flexible reporting—by staff type, trust unit, or summary rollup across all metrics and periods.


In [4]:
group_cols = ["service group", "department", "organisation", "staff group"]

In [5]:
# Leavers: Count per group per month
leavers_agg = leavers.groupby(group_cols + ["month"]).size().reset_index(name="Count")
leavers_agg["metric"] = "Leavers"

# Hires: Count per group per month
hires_agg = (
    hires.groupby(group_cols + ["month"])
    .agg(Count=("employee number", "count"))
    .reset_index()
)
hires_agg["metric"] = "Hires"

# Sickness: Sum Abs (FTE) per group per month
sickness_agg = (
    sickness.groupby(group_cols + ["month"])["abs (fte)"]
    .sum()
    .reset_index(name="Count")
)
sickness_agg["metric"] = "Sickness"

## 5. 🔗 Combine All Aggregated Data
>**Note**

After this step, `all_agg` is a single “tidy” long table, one row per group/metric/month.

In [6]:
all_agg = pd.concat([leavers_agg, hires_agg, sickness_agg], axis=0, ignore_index=True)

In [7]:
all_agg.columns = all_agg.columns.str.lower().str.strip()
(all_agg.head())

,service group,department,organisation,staff group,month,count,metric
0,CYP & Families Services,Acute Paediatrics,Childrens Day Surgery Unit 43030,Additional Clinical Services,2022-04-01,1.0,Leavers
1,CYP & Families Services,Acute Paediatrics,Childrens Day Surgery Unit 43030,Nursing and Midwifery Registered,2022-08-01,1.0,Leavers
2,CYP & Families Services,Acute Paediatrics,Childrens Inpatient Unit 43010,Additional Clinical Services,2021-10-01,1.0,Leavers
3,CYP & Families Services,Acute Paediatrics,Childrens Inpatient Unit 43010,Additional Clinical Services,2023-02-01,1.0,Leavers
4,CYP & Families Services,Acute Paediatrics,Childrens Inpatient Unit 43010,Additional Clinical Services,2023-09-01,1.0,Leavers


## 6. 💾 Export Combined Data for Analysis

**Result:**  
You now have a unified, group/month-indexed longitudinal table for Leavers, Hires, and Sickness. This dataset is now ready for time series modeling, decomposition, and dashboard reporting.                                                                                                                          


In [9]:
all_agg.to_csv(
    os.path.join(base_dir, "combined_workforce_monthly_New.csv"), index=False
)
print("Aggregation ready for decomposition")

Aggregation ready for decomposition


## 7. ⚠️ To Do & Quality Checklist
To Do
- [ ] Validate all source CSVs for missing dates or double-counted months.
- [ ] Check for underpopulated or all zero groups and consider filtering before forecasting.
- [ ] Annotate any major events (e.g., pandemic, HR process change) that may affect staff flows.